<a href="https://colab.research.google.com/github/LoneWolf206/sentiment-analyzer/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json again

In [8]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip amazon-fine-food-reviews.zip

mkdir: cannot create directory ‘/root/.kaggle’: File exists
Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
amazon-fine-food-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  amazon-fine-food-reviews.zip
replace Reviews.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace database.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hashes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [9]:
!unzip amazon-fine-food-reviews.zip

Archive:  amazon-fine-food-reviews.zip
replace Reviews.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace database.sqlite? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace hashes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [10]:
import pandas as pd
df = pd.read_csv('Reviews.csv')
print(df.shape)
print(df.head())

(568454, 10)
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality can

In [11]:
print(df['Score'].value_counts())
print(df['Text'].isnull().sum())


Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64
0


In [12]:
print(df.describe())
print(df.info())

                  Id  HelpfulnessNumerator  HelpfulnessDenominator  \
count  568454.000000         568454.000000            568454.00000   
mean   284227.500000              1.743817                 2.22881   
std    164098.679298              7.636513                 8.28974   
min         1.000000              0.000000                 0.00000   
25%    142114.250000              0.000000                 0.00000   
50%    284227.500000              0.000000                 1.00000   
75%    426340.750000              2.000000                 2.00000   
max    568454.000000            866.000000               923.00000   

               Score          Time  
count  568454.000000  5.684540e+05  
mean        4.183199  1.296257e+09  
std         1.310436  4.804331e+07  
min         1.000000  9.393408e+08  
25%         4.000000  1.271290e+09  
50%         5.000000  1.311120e+09  
75%         5.000000  1.332720e+09  
max         5.000000  1.351210e+09  
<class 'pandas.core.frame.DataFrame'

In [13]:
# Convert scores to sentiment labels
def get_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(get_sentiment)
print(df['sentiment'].value_counts())

sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64


In [14]:

# Sample equal numbers from each class
min_count = df['sentiment'].value_counts().min()
df_balanced = df.groupby('sentiment').sample(n=min_count, random_state=42)

print(df_balanced['sentiment'].value_counts())
print(df_balanced.shape)

sentiment
negative    42640
neutral     42640
positive    42640
Name: count, dtype: int64
(127920, 11)


In [15]:
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove special characters
    text = text.lower().strip()               # lowercase
    return text

df_balanced['clean_text'] = df_balanced['Text'].apply(clean_text)
print(df_balanced['clean_text'].head())

525327    i have an absolute passion for deep dark hot c...
75760     this drink is so super energy its almost frigh...
468100    im sticking with what used to be carnation now...
71864     aspertame causes alot of problems including pr...
211592    i ordered these because my local pet store sto...
Name: clean_text, dtype: object


In [16]:
df_sample = df_balanced.sample(n=15000, random_state=42)
print(df_sample['sentiment'].value_counts())

sentiment
negative    5115
neutral     5015
positive    4870
Name: count, dtype: int64
